# Attaques Adversariales sur les Systèmes de Détection d'Intrusions

**Jeu de données :** NSL-KDD  
**Modèle :** Perceptron Multi-Couches (PyTorch)  
**Attaques :** Fast Gradient Sign Method (FGSM), Projected Gradient Descent (PGD)  
**Défenses :** Entraînement adversarial, Feature Squeezing  

---

Ce notebook étudie la robustesse d'un système de détection d'intrusions (IDS) basé sur un réseau de neurones face à des perturbations adversariales. Nous évaluons des attaques en boîte blanche (*white-box*), dans l'hypothèse où l'adversaire dispose d'une connaissance complète de l'architecture et des paramètres du modèle. Deux stratégies de défense sont ensuite comparées.

**Structure du notebook :**
1. Chargement et analyse exploratoire du jeu de données
2. Prétraitement
3. Entraînement et évaluation du modèle MLP de référence
4. Attaques adversariales FGSM et PGD
5. Mécanismes de défense
6. Analyse comparative

In [ ]:
# Installation des dépendances
# Sur Google Colab, torch/sklearn/pandas/matplotlib sont pre-installes.
# Cette cellule installe uniquement les paquets manquants.
import importlib.util, subprocess, sys

PAQUETS = {
    'torch'      : 'torch',
    'sklearn'    : 'scikit-learn',
    'pandas'     : 'pandas',
    'numpy'      : 'numpy',
    'matplotlib' : 'matplotlib',
    'seaborn'    : 'seaborn',
}

manquants = [
    pip_nom for import_nom, pip_nom in PAQUETS.items()
    if importlib.util.find_spec(import_nom) is None
]

if manquants:
    print(f'Installation de : {manquants}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + manquants)
    print('Installation terminee. Faites Runtime > Restart and run all pour continuer.')
else:
    print('Tous les paquets sont deja disponibles.')

---
## 1. Chargement et Analyse Exploratoire du Jeu de Données

Le jeu de données NSL-KDD est une version améliorée du jeu de données KDD Cup 1999, corrigeant notamment les problèmes d'enregistrements redondants et de déséquilibre de classes. Il contient 41 caractéristiques de trafic réseau et une étiquette catégorielle indiquant le type de trafic (normal ou catégorie d'attaque).

Source : [Université du Nouveau-Brunswick — Jeux de données CIC](https://www.unb.ca/cic/datasets/nsl.html)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Création du répertoire de sortie des figures si inexistant
os.makedirs('../data', exist_ok=True)

print('Bibliothèques chargées avec succès.')

In [ ]:
NOMS_COLONNES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
    'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count',
    'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

In [ ]:
URL_TRAIN = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt'
URL_TEST  = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt'

df_train = pd.read_csv(URL_TRAIN, names=NOMS_COLONNES)
df_test  = pd.read_csv(URL_TEST,  names=NOMS_COLONNES)

df_train.drop(columns=['difficulty_level'], inplace=True)
df_test.drop(columns=['difficulty_level'],  inplace=True)

print(f"Ensemble d'entraînement : {df_train.shape[0]:,} échantillons, {df_train.shape[1]} colonnes")
print(f"Ensemble de test        : {df_test.shape[0]:,} échantillons, {df_test.shape[1]} colonnes")

### 1.1 Aperçu du Jeu de Données

In [ ]:
df_train.head(5)

In [ ]:
features_categoriques = df_train.select_dtypes(include='object').columns.tolist()
features_numeriques   = df_train.select_dtypes(include=np.number).columns.tolist()

print('Features catégorielles :', features_categoriques)
print(f'Features numériques    : {len(features_numeriques)} colonnes')
print(f'Valeurs manquantes     : {df_train.isnull().sum().sum()}')

### 1.2 Distribution des Classes

Les étiquettes NSL-KDD sont multi-classes : trafic `normal` et quatre catégories d'attaques — DoS, Probe, R2L, U2R. Pour une classification IDS binaire, toutes les catégories d'attaques sont regroupées sous une classe unique `attaque`.

In [ ]:
CATEGORIES_ATTAQUES = {
    'DoS'   : ['back','land','neptune','pod','smurf','teardrop','apache2',
               'udpstorm','processtable','worm'],
    'Probe' : ['satan','ipsweep','nmap','portsweep','mscan','saint'],
    'R2L'   : ['guess_passwd','ftp_write','imap','phf','multihop','warezmaster',
               'warezclient','spy','xlock','xsnoop','snmpguess','snmpgetattack',
               'httptunnel','sendmail','named'],
    'U2R'   : ['buffer_overflow','loadmodule','perl','rootkit','ps',
               'sqlattack','xterm']
}

label_vers_categorie = {}
for categorie, labels in CATEGORIES_ATTAQUES.items():
    for lbl in labels:
        label_vers_categorie[lbl] = categorie

df_train['categorie_attaque'] = df_train['label'].apply(
    lambda x: label_vers_categorie.get(x, 'Normal' if x == 'normal' else 'Autre')
)

effectifs = df_train['categorie_attaque'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

effectifs.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title("Effectifs par Catégorie d'Attaque")
axes[0].set_xlabel('Catégorie')
axes[0].set_ylabel('Effectif')
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(
    effectifs,
    labels=effectifs.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=plt.cm.Set2.colors
)
axes[1].set_title("Distribution des Classes (Ensemble d'Entraînement)")

plt.tight_layout()
plt.savefig('../data/distribution_classes.png', bbox_inches='tight')
plt.show()

print(effectifs.to_frame('effectif').assign(
    pourcentage=lambda df: (df['effectif'] / df['effectif'].sum() * 100).round(2)
))

### 1.3 Analyse des Features

In [ ]:
for col in features_categoriques[:-1]:
    print(f'{col:20s} — {df_train[col].nunique()} valeurs uniques : {df_train[col].unique().tolist()}')

In [ ]:
df_train[features_numeriques[:10]].describe().round(3)

In [ ]:
features_taux = [c for c in features_numeriques if 'rate' in c or 'count' in c]

plt.figure(figsize=(10, 8))
matrice_corr = df_train[features_taux].corr()
sns.heatmap(
    matrice_corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    annot_kws={'size': 7}
)
plt.title('Matrice de Corrélation — Features de Taux et de Comptage')
plt.tight_layout()
plt.savefig('../data/matrice_correlation.png', bbox_inches='tight')
plt.show()

---
## 2. Prétraitement

Le prétraitement comprend quatre étapes :

1. **Encodage one-hot** des features catégorielles (`protocol_type`, `service`, `flag`) — les colonnes manquantes dans l'ensemble de test sont ajoutées avec la valeur 0 pour garantir l'alignement des dimensions.
2. **Binarisation des étiquettes** — toute valeur différente de `normal` est encodée comme `1` (attaque), `normal` est encodé comme `0`.
3. **Normalisation Min-Max** — les features numériques sont ramenées dans l'intervalle [0, 1]. Le `MinMaxScaler` est ajusté uniquement sur l'ensemble d'entraînement afin d'éviter toute fuite de données (*data leakage*).
4. **Conversion en tenseurs PyTorch** — les tableaux NumPy sont convertis en tenseurs `float32` pour être compatibles avec le pipeline d'entraînement.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import torch

print(f'Version PyTorch : {torch.__version__}')

### 2.1 Encodage One-Hot des Features Catégorielles

In [ ]:
FEATURES_CATEGORIELLES = ['protocol_type', 'service', 'flag']

df_train_enc = pd.get_dummies(df_train, columns=FEATURES_CATEGORIELLES)
df_test_enc  = pd.get_dummies(df_test,  columns=FEATURES_CATEGORIELLES)

# Alignement des colonnes : le test peut manquer certaines modalités présentes dans le train
colonnes_train = [c for c in df_train_enc.columns if c not in ('label', 'categorie_attaque')]
df_test_enc = df_test_enc.reindex(columns=colonnes_train, fill_value=0)

print(f'Dimensions après encodage — entraînement : {df_train_enc.shape}')
print(f'Dimensions après encodage — test         : {df_test_enc.shape}')

### 2.2 Binarisation des Étiquettes

In [ ]:
y_train_bin = (df_train['label'] != 'normal').astype(int).values
y_test_bin  = (df_test['label']  != 'normal').astype(int).values

print(f"Entraînement — normal : {(y_train_bin == 0).sum():,}  |  attaque : {(y_train_bin == 1).sum():,}")
print(f"Test          — normal : {(y_test_bin  == 0).sum():,}  |  attaque : {(y_test_bin  == 1).sum():,}")

### 2.3 Normalisation Min-Max

In [ ]:
COLONNES_FEATURES = [c for c in colonnes_train if c not in ('label', 'categorie_attaque')]

X_train_raw = df_train_enc[COLONNES_FEATURES].values.astype(np.float32)
X_test_raw  = df_test_enc[COLONNES_FEATURES].values.astype(np.float32)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print(f'Plage des valeurs après normalisation — min : {X_train_scaled.min():.4f}  |  max : {X_train_scaled.max():.4f}')
print(f'Dimension du vecteur de features : {X_train_scaled.shape[1]}')

### 2.4 Découpage Entraînement / Validation

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_scaled, y_train_bin,
    test_size=0.15,
    random_state=42,
    stratify=y_train_bin
)

print(f'Entraînement : {X_train.shape[0]:,} échantillons')
print(f'Validation   : {X_val.shape[0]:,} échantillons')
print(f'Test         : {X_test_scaled.shape[0]:,} échantillons')

### 2.5 Conversion en Tenseurs PyTorch

In [ ]:
X_train_t = torch.tensor(X_train,        dtype=torch.float32)
y_train_t = torch.tensor(y_train,        dtype=torch.float32)
X_val_t   = torch.tensor(X_val,          dtype=torch.float32)
y_val_t   = torch.tensor(y_val,          dtype=torch.float32)
X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test_bin,     dtype=torch.float32)

DIM_ENTREE = X_train_t.shape[1]

print(f'Dimension du tenseur d\'entrée : {DIM_ENTREE}')
print(f'X_train_t : {X_train_t.shape}  |  y_train_t : {y_train_t.shape}')
print(f'X_val_t   : {X_val_t.shape}  |  y_val_t   : {y_val_t.shape}')
print(f'X_test_t  : {X_test_t.shape}   |  y_test_t  : {y_test_t.shape}')

---
## 3. Modèle MLP de Référence

Nous définissons un perceptron multi-couches (MLP) comme classifieur de base. L'architecture retenue est volontairement sobre afin de rester interprétable et de rendre les attaques adversariales calculables efficacement sur CPU.

**Architecture :**
- Couche d'entrée : `DIM_ENTREE` neurones
- Couche cachée 1 : 128 neurones, activation ReLU, Dropout 0.3
- Couche cachée 2 : 64 neurones, activation ReLU, Dropout 0.3
- Couche cachée 3 : 32 neurones, activation ReLU
- Couche de sortie : 1 neurone, activation Sigmoid (classification binaire)

**Fonction de perte :** `BCELoss` (Binary Cross-Entropy)  
**Optimiseur :** Adam, taux d'apprentissage `1e-3`  
**Arrêt anticipé :** patience de 10 époques sur la perte de validation

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

GRAINE = 42
torch.manual_seed(GRAINE)
np.random.seed(GRAINE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositif utilisé : {DEVICE}')

### 3.1 Définition de l'Architecture

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim_entree: int, taux_dropout: float = 0.3):
        super(MLP, self).__init__()
        self.reseau = nn.Sequential(
            nn.Linear(dim_entree, 128),
            nn.ReLU(),
            nn.Dropout(taux_dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(taux_dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.reseau(x).squeeze(1)


modele = MLP(dim_entree=DIM_ENTREE).to(DEVICE)
print(modele)

nb_parametres = sum(p.numel() for p in modele.parameters() if p.requires_grad)
print(f'\nNombre de paramètres entraînables : {nb_parametres:,}')

### 3.2 Entraînement

In [ ]:
TAILLE_BATCH  = 512
NB_EPOQUES    = 100
PATIENCE      = 10
TAUX_APPRENTISSAGE = 1e-3

dataset_train = TensorDataset(X_train_t.to(DEVICE), y_train_t.to(DEVICE))
dataset_val   = TensorDataset(X_val_t.to(DEVICE),   y_val_t.to(DEVICE))

chargeur_train = DataLoader(dataset_train, batch_size=TAILLE_BATCH, shuffle=True)
chargeur_val   = DataLoader(dataset_val,   batch_size=TAILLE_BATCH, shuffle=False)

critere    = nn.BCELoss()
optimiseur = optim.Adam(modele.parameters(), lr=TAUX_APPRENTISSAGE)

historique = {'perte_train': [], 'perte_val': [], 'acc_train': [], 'acc_val': []}
meilleure_perte_val = float('inf')
compteur_patience   = 0
meilleur_etat       = None

for epoque in range(1, NB_EPOQUES + 1):
    # Phase d'entraînement
    modele.train()
    perte_train_ep, correct_train, total_train = 0.0, 0, 0
    for X_batch, y_batch in chargeur_train:
        optimiseur.zero_grad()
        predictions = modele(X_batch)
        perte = critere(predictions, y_batch)
        perte.backward()
        optimiseur.step()
        perte_train_ep += perte.item() * len(y_batch)
        correct_train  += ((predictions >= 0.5).float() == y_batch).sum().item()
        total_train    += len(y_batch)

    # Phase de validation
    modele.eval()
    perte_val_ep, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in chargeur_val:
            predictions = modele(X_batch)
            perte = critere(predictions, y_batch)
            perte_val_ep += perte.item() * len(y_batch)
            correct_val  += ((predictions >= 0.5).float() == y_batch).sum().item()
            total_val    += len(y_batch)

    perte_train_moy = perte_train_ep / total_train
    perte_val_moy   = perte_val_ep   / total_val
    acc_train_moy   = correct_train  / total_train
    acc_val_moy     = correct_val    / total_val

    historique['perte_train'].append(perte_train_moy)
    historique['perte_val'].append(perte_val_moy)
    historique['acc_train'].append(acc_train_moy)
    historique['acc_val'].append(acc_val_moy)

    if epoque % 10 == 0:
        print(f'Époque {epoque:3d}/{NB_EPOQUES}  '
              f'perte_train={perte_train_moy:.4f}  perte_val={perte_val_moy:.4f}  '
              f'acc_train={acc_train_moy:.4f}  acc_val={acc_val_moy:.4f}')

    # Arrêt anticipé
    if perte_val_moy < meilleure_perte_val:
        meilleure_perte_val = perte_val_moy
        compteur_patience   = 0
        meilleur_etat       = {k: v.clone() for k, v in modele.state_dict().items()}
    else:
        compteur_patience += 1
        if compteur_patience >= PATIENCE:
            print(f'Arrêt anticipé à l\'époque {epoque}.')
            break

modele.load_state_dict(meilleur_etat)
print('Meilleur modèle restauré.')

### 3.3 Courbes d'Apprentissage

In [ ]:
epoques = range(1, len(historique['perte_train']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epoques, historique['perte_train'], label='Entraînement', color='steelblue')
axes[0].plot(epoques, historique['perte_val'],   label='Validation',   color='tomato', linestyle='--')
axes[0].set_title('Évolution de la Perte (BCE)')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Perte')
axes[0].legend()

axes[1].plot(epoques, historique['acc_train'], label='Entraînement', color='steelblue')
axes[1].plot(epoques, historique['acc_val'],   label='Validation',   color='tomato', linestyle='--')
axes[1].set_title('Évolution de la Précision')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Précision')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/courbes_apprentissage.png', bbox_inches='tight')
plt.show()

### 3.4 Évaluation sur l'Ensemble de Test

In [ ]:
def evaluer_modele(modele, X_t, y_t, nom_ensemble='Test'):
    modele.eval()
    with torch.no_grad():
        predictions_brutes = modele(X_t.to(DEVICE)).cpu().numpy()
    predictions_bin = (predictions_brutes >= 0.5).astype(int)
    y_np = y_t.numpy().astype(int)

    metriques = {
        'Précision'  : accuracy_score(y_np,  predictions_bin),
        'Exactitude' : precision_score(y_np, predictions_bin, zero_division=0),
        'Rappel'     : recall_score(y_np,    predictions_bin, zero_division=0),
        'F1-score'   : f1_score(y_np,        predictions_bin, zero_division=0),
    }

    print(f'\n=== Évaluation — {nom_ensemble} ===')
    for nom, valeur in metriques.items():
        print(f'  {nom:12s} : {valeur:.4f}')

    print(f'\nRapport de classification :\n')
    print(classification_report(y_np, predictions_bin,
                                target_names=['Normal', 'Attaque'], zero_division=0))

    mat_conf = confusion_matrix(y_np, predictions_bin)
    plt.figure(figsize=(5, 4))
    sns.heatmap(mat_conf, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Attaque'],
                yticklabels=['Normal', 'Attaque'])
    plt.title(f'Matrice de Confusion — {nom_ensemble}')
    plt.ylabel('Étiquette réelle')
    plt.xlabel('Étiquette prédite')
    plt.tight_layout()
    plt.savefig(f'../data/confusion_{nom_ensemble.lower()}.png', bbox_inches='tight')
    plt.show()

    return metriques


metriques_reference = evaluer_modele(modele, X_test_t, y_test_t, nom_ensemble='Test (référence)')

---
## 4. Attaques Adversariales

Les attaques adversariales consistent à introduire des perturbations imperceptibles dans les données d'entrée afin de tromper le modèle. Dans le contexte d'un IDS, un adversaire cherche à faire passer du trafic malveillant pour du trafic normal.

Nous implémentons deux attaques en boîte blanche (*white-box*) :

- **FGSM** (*Fast Gradient Sign Method*, Goodfellow et al., 2014) : perturbation en un seul pas dans la direction du gradient de la perte par rapport à l'entrée.
- **PGD** (*Projected Gradient Descent*, Madry et al., 2018) : version itérative de FGSM avec projection dans une boule $\ell_\infty$ de rayon $\varepsilon$, considérée comme l'attaque de référence en robustesse adversariale.

**Contrainte :** les perturbations sont bornées par $\varepsilon$ en norme $\ell_\infty$ et les exemples adversariaux sont maintenus dans $[0, 1]$ après chaque perturbation.

### 4.1 FGSM — Fast Gradient Sign Method

La perturbation FGSM est définie par :

$$x_{adv} = x + \varepsilon \cdot \text{sign}\left(\nabla_x \mathcal{L}(f_\theta(x), y)\right)$$

où $\mathcal{L}$ est la fonction de perte, $f_\theta$ le modèle et $\varepsilon$ le budget de perturbation.

In [ ]:
def fgsm(modele, X, y, epsilon, critere):
    """Génère des exemples adversariaux par la méthode FGSM.

    Paramètres
    ----------
    modele  : nn.Module      — modèle cible
    X       : torch.Tensor   — exemples d'entrée dans [0, 1]
    y       : torch.Tensor   — étiquettes binaires
    epsilon : float          — budget de perturbation (norme L-inf)
    critere : nn.Module      — fonction de perte

    Retourne
    --------
    torch.Tensor : exemples adversariaux dans [0, 1]
    """
    X_adv = X.clone().detach().requires_grad_(True)
    predictions = modele(X_adv)
    perte = critere(predictions, y)
    modele.zero_grad()
    perte.backward()
    perturbation = epsilon * X_adv.grad.sign()
    X_adv = X_adv.detach() + perturbation
    X_adv = torch.clamp(X_adv, 0.0, 1.0)
    return X_adv.detach()

In [ ]:
VALEURS_EPSILON = [0.01, 0.05, 0.1, 0.2, 0.3]

resultats_fgsm = {}

modele.eval()
print('=== Évaluation FGSM par valeur d\'epsilon ===\n')

for eps in VALEURS_EPSILON:
    X_adv = fgsm(modele, X_test_t.to(DEVICE), y_test_t.to(DEVICE), eps, critere)
    with torch.no_grad():
        preds = (modele(X_adv).cpu().numpy() >= 0.5).astype(int)
    acc = accuracy_score(y_test_t.numpy().astype(int), preds)
    f1  = f1_score(y_test_t.numpy().astype(int), preds, zero_division=0)
    resultats_fgsm[eps] = {'accuracy': acc, 'f1': f1}
    print(f'  epsilon={eps:.2f}  accuracy={acc:.4f}  f1={f1:.4f}')

# Visualisation de la dégradation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eps_vals  = list(resultats_fgsm.keys())
acc_vals  = [resultats_fgsm[e]['accuracy'] for e in eps_vals]
f1_vals   = [resultats_fgsm[e]['f1']       for e in eps_vals]

axes[0].plot(eps_vals, acc_vals, marker='o', color='steelblue')
axes[0].axhline(metriques_reference['Précision'], color='gray', linestyle='--', label='Référence')
axes[0].set_title('FGSM — Dégradation de la Précision')
axes[0].set_xlabel('Epsilon')
axes[0].set_ylabel('Précision')
axes[0].legend()

axes[1].plot(eps_vals, f1_vals, marker='o', color='tomato')
axes[1].axhline(metriques_reference['F1-score'], color='gray', linestyle='--', label='Référence')
axes[1].set_title('FGSM — Dégradation du F1-score')
axes[1].set_xlabel('Epsilon')
axes[1].set_ylabel('F1-score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/fgsm_degradation.png', bbox_inches='tight')
plt.show()

# Conservation des exemples adversariaux pour epsilon de référence
EPSILON_REF = 0.1
X_fgsm_ref = fgsm(modele, X_test_t.to(DEVICE), y_test_t.to(DEVICE), EPSILON_REF, critere)

### 4.2 PGD — Projected Gradient Descent

PGD est une version itérative de FGSM. À chaque itération $t$, la perturbation est mise à jour puis projetée dans la boule $\ell_\infty(\varepsilon)$ :

$$x^{(t+1)}_{adv} = \Pi_{\mathcal{B}_\varepsilon(x)} \left( x^{(t)}_{adv} + \alpha \cdot \text{sign}\left(\nabla_{x^{(t)}_{adv}} \mathcal{L}\right) \right)$$

où $\alpha$ est le pas par itération et $\Pi$ est l'opérateur de projection. Le point de départ est perturbé aléatoirement (*random start*).

In [ ]:
def pgd(modele, X, y, epsilon, alpha, nb_iterations, critere):
    """Génère des exemples adversariaux par la méthode PGD.

    Paramètres
    ----------
    modele        : nn.Module      — modèle cible
    X             : torch.Tensor   — exemples d'entrée dans [0, 1]
    y             : torch.Tensor   — étiquettes binaires
    epsilon       : float          — budget de perturbation (norme L-inf)
    alpha         : float          — pas par itération
    nb_iterations : int            — nombre d'itérations
    critere       : nn.Module      — fonction de perte

    Retourne
    --------
    torch.Tensor : exemples adversariaux dans [0, 1]
    """
    # Initialisation aléatoire dans la boule epsilon
    X_adv = X.clone().detach()
    X_adv = X_adv + torch.empty_like(X_adv).uniform_(-epsilon, epsilon)
    X_adv = torch.clamp(X_adv, 0.0, 1.0).detach()

    for _ in range(nb_iterations):
        X_adv.requires_grad_(True)
        predictions = modele(X_adv)
        perte = critere(predictions, y)
        modele.zero_grad()
        perte.backward()

        with torch.no_grad():
            X_adv = X_adv + alpha * X_adv.grad.sign()
            # Projection dans la boule L-inf
            delta = torch.clamp(X_adv - X, -epsilon, epsilon)
            X_adv = torch.clamp(X + delta, 0.0, 1.0).detach()

    return X_adv

In [ ]:
CONFIGS_PGD = [
    {'epsilon': 0.01, 'alpha': 0.005, 'nb_iterations': 10},
    {'epsilon': 0.05, 'alpha': 0.01,  'nb_iterations': 20},
    {'epsilon': 0.1,  'alpha': 0.01,  'nb_iterations': 40},
    {'epsilon': 0.2,  'alpha': 0.02,  'nb_iterations': 40},
    {'epsilon': 0.3,  'alpha': 0.03,  'nb_iterations': 40},
]

resultats_pgd = {}

modele.eval()
print('=== Évaluation PGD par configuration ===\n')

for config in CONFIGS_PGD:
    eps = config['epsilon']
    X_adv = pgd(
        modele,
        X_test_t.to(DEVICE), y_test_t.to(DEVICE),
        epsilon=eps,
        alpha=config['alpha'],
        nb_iterations=config['nb_iterations'],
        critere=critere
    )
    with torch.no_grad():
        preds = (modele(X_adv).cpu().numpy() >= 0.5).astype(int)
    acc = accuracy_score(y_test_t.numpy().astype(int), preds)
    f1  = f1_score(y_test_t.numpy().astype(int), preds, zero_division=0)
    resultats_pgd[eps] = {'accuracy': acc, 'f1': f1}
    print(f'  epsilon={eps:.2f}  iterations={config["nb_iterations"]:2d}  '
          f'accuracy={acc:.4f}  f1={f1:.4f}')

# Visualisation comparative FGSM vs PGD
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eps_communs = [e for e in VALEURS_EPSILON if e in resultats_pgd]

axes[0].plot(eps_communs, [resultats_fgsm[e]['accuracy'] for e in eps_communs],
             marker='o', label='FGSM', color='steelblue')
axes[0].plot(eps_communs, [resultats_pgd[e]['accuracy']  for e in eps_communs],
             marker='s', label='PGD',  color='tomato')
axes[0].axhline(metriques_reference['Précision'], color='gray', linestyle='--', label='Référence')
axes[0].set_title('Comparaison FGSM / PGD — Précision')
axes[0].set_xlabel('Epsilon')
axes[0].set_ylabel('Précision')
axes[0].legend()

axes[1].plot(eps_communs, [resultats_fgsm[e]['f1'] for e in eps_communs],
             marker='o', label='FGSM', color='steelblue')
axes[1].plot(eps_communs, [resultats_pgd[e]['f1']  for e in eps_communs],
             marker='s', label='PGD',  color='tomato')
axes[1].axhline(metriques_reference['F1-score'], color='gray', linestyle='--', label='Référence')
axes[1].set_title('Comparaison FGSM / PGD — F1-score')
axes[1].set_xlabel('Epsilon')
axes[1].set_ylabel('F1-score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/comparaison_fgsm_pgd.png', bbox_inches='tight')
plt.show()

# Conservation des exemples adversariaux PGD pour epsilon de référence
X_pgd_ref = pgd(
    modele,
    X_test_t.to(DEVICE), y_test_t.to(DEVICE),
    epsilon=EPSILON_REF, alpha=0.01, nb_iterations=40,
    critere=critere
)

print('\n--- Évaluation sur exemples adversariaux (epsilon=0.1) ---')
metriques_fgsm = evaluer_modele(modele, X_fgsm_ref.cpu(), y_test_t, nom_ensemble='FGSM (eps=0.1)')
metriques_pgd  = evaluer_modele(modele, X_pgd_ref.cpu(),  y_test_t, nom_ensemble='PGD  (eps=0.1)')

---
## 5. Mécanismes de Défense

Deux stratégies de défense sont évaluées :

- **Entraînement adversarial** (*Adversarial Training*) : le modèle est réentraîné sur un mélange d'exemples propres et d'exemples adversariaux générés à la volée. Cette approche, introduite par Madry et al. (2018), est considérée comme la défense empirique la plus robuste à ce jour.
- **Feature Squeezing** : les features d'entrée sont lissées avant d'être présentées au modèle, réduisant ainsi l'espace dans lequel les perturbations adversariales peuvent exister. Deux variantes sont testées : arrondi à faible précision (*bit reduction*) et lissage par médiane.

Les deux défenses sont évaluées sur les mêmes exemples adversariaux FGSM et PGD générés à l'étape précédente (`epsilon=0.1`).

### 5.1 Entraînement Adversarial

À chaque batch, la moitié des exemples est remplacée par des exemples adversariaux FGSM générés à la volée. Le modèle apprend ainsi à classer correctement aussi bien les exemples propres que les exemples perturbés.

In [ ]:
torch.manual_seed(GRAINE)
modele_robuste = MLP(dim_entree=DIM_ENTREE).to(DEVICE)
optimiseur_robuste = optim.Adam(modele_robuste.parameters(), lr=TAUX_APPRENTISSAGE)

historique_robuste = {'perte_train': [], 'perte_val': [], 'acc_train': [], 'acc_val': []}
meilleure_perte_val_r = float('inf')
compteur_patience_r   = 0
meilleur_etat_r       = None

EPSILON_TRAIN = 0.1

for epoque in range(1, NB_EPOQUES + 1):
    modele_robuste.train()
    perte_train_ep, correct_train, total_train = 0.0, 0, 0

    for X_batch, y_batch in chargeur_train:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        # Génération d'exemples adversariaux FGSM sur la moitié du batch
        taille_adv = len(X_batch) // 2
        X_adv_batch = fgsm(modele_robuste, X_batch[:taille_adv],
                           y_batch[:taille_adv], EPSILON_TRAIN, critere)
        X_mixte = torch.cat([X_adv_batch, X_batch[taille_adv:]], dim=0)
        y_mixte = y_batch

        optimiseur_robuste.zero_grad()
        predictions = modele_robuste(X_mixte)
        perte = critere(predictions, y_mixte)
        perte.backward()
        optimiseur_robuste.step()

        perte_train_ep += perte.item() * len(y_mixte)
        correct_train  += ((predictions >= 0.5).float() == y_mixte).sum().item()
        total_train    += len(y_mixte)

    modele_robuste.eval()
    perte_val_ep, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in chargeur_val:
            predictions = modele_robuste(X_batch.to(DEVICE))
            perte = critere(predictions, y_batch.to(DEVICE))
            perte_val_ep += perte.item() * len(y_batch)
            correct_val  += ((predictions >= 0.5).float() == y_batch.to(DEVICE)).sum().item()
            total_val    += len(y_batch)

    perte_train_moy = perte_train_ep / total_train
    perte_val_moy   = perte_val_ep   / total_val
    acc_train_moy   = correct_train  / total_train
    acc_val_moy     = correct_val    / total_val

    historique_robuste['perte_train'].append(perte_train_moy)
    historique_robuste['perte_val'].append(perte_val_moy)
    historique_robuste['acc_train'].append(acc_train_moy)
    historique_robuste['acc_val'].append(acc_val_moy)

    if epoque % 10 == 0:
        print(f'Époque {epoque:3d}/{NB_EPOQUES}  '
              f'perte_train={perte_train_moy:.4f}  perte_val={perte_val_moy:.4f}  '
              f'acc_val={acc_val_moy:.4f}')

    if perte_val_moy < meilleure_perte_val_r:
        meilleure_perte_val_r = perte_val_moy
        compteur_patience_r   = 0
        meilleur_etat_r = {k: v.clone() for k, v in modele_robuste.state_dict().items()}
    else:
        compteur_patience_r += 1
        if compteur_patience_r >= PATIENCE:
            print(f'Arrêt anticipé à l\'époque {epoque}.')
            break

modele_robuste.load_state_dict(meilleur_etat_r)
print('Modèle robuste restauré.')

In [ ]:
print('--- Évaluation du modèle robuste (entraînement adversarial) ---')
metriques_robuste_propre = evaluer_modele(
    modele_robuste, X_test_t, y_test_t, nom_ensemble='Robuste — exemples propres'
)
metriques_robuste_fgsm = evaluer_modele(
    modele_robuste, X_fgsm_ref.cpu(), y_test_t, nom_ensemble='Robuste — FGSM (eps=0.1)'
)
metriques_robuste_pgd = evaluer_modele(
    modele_robuste, X_pgd_ref.cpu(), y_test_t, nom_ensemble='Robuste — PGD (eps=0.1)'
)

### 5.2 Feature Squeezing

Le *feature squeezing* réduit la précision des features d'entrée afin de supprimer les perturbations à faible amplitude. Deux transformations sont appliquées au modèle de référence (non robuste) :

- **Arrondi** (*bit reduction*) : les valeurs sont arrondies à `n` décimales significatives.
- **Lissage par médiane** : chaque feature est remplacée par la médiane de son voisinage (fenêtre glissante de taille `k`).

In [ ]:
def arrondi(X: torch.Tensor, nb_decimales: int) -> torch.Tensor:
    """Réduit la précision des features par arrondi à nb_decimales décimales."""
    facteur = 10 ** nb_decimales
    return torch.round(X * facteur) / facteur


def lissage_mediane(X: torch.Tensor, taille_fenetre: int) -> torch.Tensor:
    """Applique un lissage par médiane sur les features (fenêtre glissante).

    Paramètres
    ----------
    X             : torch.Tensor — tenseur (N, D)
    taille_fenetre : int          — largeur de la fenêtre (doit être impaire)

    Retourne
    --------
    torch.Tensor : tenseur lissé de même dimension
    """
    X_np = X.numpy()
    demi = taille_fenetre // 2
    X_lisse = np.copy(X_np)
    for j in range(X_np.shape[1]):
        debut = max(0, j - demi)
        fin   = min(X_np.shape[1], j + demi + 1)
        X_lisse[:, j] = np.median(X_np[:, debut:fin], axis=1)
    return torch.tensor(X_lisse, dtype=torch.float32)

In [ ]:
configurations_fs = [
    ('Arrondi 1 décimale',  lambda X: arrondi(X, 1)),
    ('Arrondi 2 décimales', lambda X: arrondi(X, 2)),
    ('Médiane fenêtre=3',   lambda X: lissage_mediane(X, 3)),
    ('Médiane fenêtre=5',   lambda X: lissage_mediane(X, 5)),
]

resultats_fs = {}

print('=== Feature Squeezing — évaluation sur exemples adversariaux (eps=0.1) ===\n')
print(f'{"Configuration":25s}  {"Données":10s}  {"Précision":>10s}  {"F1-score":>10s}')
print('-' * 62)

for nom, transformateur in configurations_fs:
    resultats_fs[nom] = {}
    for label_donnees, X_eval in [
        ('Propres', X_test_t),
        ('FGSM',    X_fgsm_ref.cpu()),
        ('PGD',     X_pgd_ref.cpu()),
    ]:
        X_lisse = transformateur(X_eval)
        modele.eval()
        with torch.no_grad():
            preds = (modele(X_lisse.to(DEVICE)).cpu().numpy() >= 0.5).astype(int)
        acc = accuracy_score(y_test_t.numpy().astype(int), preds)
        f1  = f1_score(y_test_t.numpy().astype(int), preds, zero_division=0)
        resultats_fs[nom][label_donnees] = {'accuracy': acc, 'f1': f1}
        print(f'{nom:25s}  {label_donnees:10s}  {acc:10.4f}  {f1:10.4f}')

---
## 6. Analyse Comparative et Synthèse

Cette section consolide l'ensemble des résultats obtenus au cours des étapes précédentes afin d'évaluer objectivement l'impact des attaques adversariales et l'efficacité des défenses. Trois dimensions sont analysées :

1. Dégradation des performances sous attaque (FGSM et PGD à `epsilon=0.1`)
2. Récupération de robustesse par l'entraînement adversarial
3. Efficacité comparative du feature squeezing

### 6.1 Tableau Récapitulatif Global

In [ ]:
lignes = [
    {
        'Modèle'   : 'MLP standard',
        'Données'  : 'Propres',
        'Précision': metriques_reference['Précision'],
        'F1-score' : metriques_reference['F1-score'],
    },
    {
        'Modèle'   : 'MLP standard',
        'Données'  : 'FGSM (eps=0.1)',
        'Précision': metriques_fgsm['Précision'],
        'F1-score' : metriques_fgsm['F1-score'],
    },
    {
        'Modèle'   : 'MLP standard',
        'Données'  : 'PGD (eps=0.1)',
        'Précision': metriques_pgd['Précision'],
        'F1-score' : metriques_pgd['F1-score'],
    },
    {
        'Modèle'   : 'Entraînement adversarial',
        'Données'  : 'Propres',
        'Précision': metriques_robuste_propre['Précision'],
        'F1-score' : metriques_robuste_propre['F1-score'],
    },
    {
        'Modèle'   : 'Entraînement adversarial',
        'Données'  : 'FGSM (eps=0.1)',
        'Précision': metriques_robuste_fgsm['Précision'],
        'F1-score' : metriques_robuste_fgsm['F1-score'],
    },
    {
        'Modèle'   : 'Entraînement adversarial',
        'Données'  : 'PGD (eps=0.1)',
        'Précision': metriques_robuste_pgd['Précision'],
        'F1-score' : metriques_robuste_pgd['F1-score'],
    },
]

# Ajout des résultats feature squeezing (meilleure configuration par attaque)
for nom_fs, res_fs in resultats_fs.items():
    for label_donnees in ['Propres', 'FGSM', 'PGD']:
        label_affiche = label_donnees if label_donnees == 'Propres' else f'{label_donnees} (eps=0.1)'
        lignes.append({
            'Modèle'   : f'Feature Squeezing — {nom_fs}',
            'Données'  : label_affiche,
            'Précision': res_fs[label_donnees]['accuracy'],
            'F1-score' : res_fs[label_donnees]['f1'],
        })

df_resultats = pd.DataFrame(lignes)
df_resultats['Précision'] = df_resultats['Précision'].round(4)
df_resultats['F1-score']  = df_resultats['F1-score'].round(4)

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', 40)
df_resultats

### 6.2 Impact des Attaques sur le Modèle Standard

In [ ]:
categories  = ['Propres', 'FGSM (eps=0.1)', 'PGD (eps=0.1)']
acc_std     = [
    metriques_reference['Précision'],
    metriques_fgsm['Précision'],
    metriques_pgd['Précision'],
]
acc_robuste = [
    metriques_robuste_propre['Précision'],
    metriques_robuste_fgsm['Précision'],
    metriques_robuste_pgd['Précision'],
]

x     = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
barres_std     = ax.bar(x - width/2, acc_std,     width, label='MLP standard',            color='steelblue')
barres_robuste = ax.bar(x + width/2, acc_robuste, width, label='Entraînement adversarial', color='seagreen')

ax.set_title('Précision — MLP Standard vs Entraînement Adversarial')
ax.set_ylabel('Précision')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0, 1.05)
ax.legend()

for barre in list(barres_std) + list(barres_robuste):
    ax.text(
        barre.get_x() + barre.get_width() / 2,
        barre.get_height() + 0.01,
        f'{barre.get_height():.3f}',
        ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()
plt.savefig('../data/comparaison_defenses.png', bbox_inches='tight')
plt.show()

### 6.3 Efficacité du Feature Squeezing sous Attaque PGD

In [ ]:
noms_fs  = list(resultats_fs.keys())
acc_pgd_fs = [resultats_fs[n]['PGD']['accuracy'] for n in noms_fs]
f1_pgd_fs  = [resultats_fs[n]['PGD']['f1']       for n in noms_fs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(noms_fs, acc_pgd_fs, color='mediumpurple', edgecolor='white')
axes[0].axhline(metriques_pgd['Précision'], color='tomato',  linestyle='--', label='Sans défense')
axes[0].axhline(metriques_reference['Précision'], color='gray', linestyle=':', label='Référence (propres)')
axes[0].set_title('Feature Squeezing — Précision sous PGD')
axes[0].set_ylabel('Précision')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(fontsize=8)

axes[1].bar(noms_fs, f1_pgd_fs, color='mediumpurple', edgecolor='white')
axes[1].axhline(metriques_pgd['F1-score'], color='tomato',  linestyle='--', label='Sans défense')
axes[1].axhline(metriques_reference['F1-score'], color='gray', linestyle=':', label='Référence (propres)')
axes[1].set_title('Feature Squeezing — F1-score sous PGD')
axes[1].set_ylabel('F1-score')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/feature_squeezing_pgd.png', bbox_inches='tight')
plt.show()

### 6.4 Conclusion

Cette étude a évalué la vulnérabilité d'un IDS neuronal face à des attaques adversariales en boîte blanche et mesuré l'efficacité de deux stratégies de défense.

**Principaux enseignements :**

- Le modèle MLP standard, bien qu'efficace sur des données propres, présente une dégradation significative de ses performances sous attaque FGSM et PGD, confirmant la vulnérabilité des IDS basés sur l'apprentissage automatique aux perturbations adversariales.

- L'attaque PGD, itérative et bornée, produit des exemples adversariaux plus puissants que FGSM : la chute de précision est systématiquement plus marquée pour un même budget $\varepsilon$.

- L'entraînement adversarial améliore significativement la robustesse du modèle face aux deux types d'attaques, avec un coût limité sur les données propres. Cette approche reste la défense empirique la plus efficace.

- Le feature squeezing constitue une défense légère ne nécessitant aucun réentraînement. Son efficacité dépend du degré de lissage appliqué : un arrondi trop agressif ou une fenêtre médiane trop large peuvent dégrader les performances sur données propres.

**Perspectives :** des attaques en boîte noire (*black-box*), des perturbations bornées en norme $\ell_2$ et des défenses basées sur la détection (détecteur d'exemples adversariaux) constituent des prolongements naturels de ce travail.